In [30]:
import pandas as pd
import numpy as np

In [31]:
ball = pd.read_csv("Datasets/Cleaned_Datasets/ball_cleaned_data.csv")
matches = pd.read_csv("Datasets/Cleaned_Datasets/matches_cleaned_data.csv")

In [32]:
# DATE HANDLING

matches["match_date"] = pd.to_datetime(matches["match_date"], errors="coerce")

ball = ball.merge(
    matches[["match_id", "match_date", "venue"]],
    on="match_id",
    how="left"
)

In [33]:
# SORTING

ball = ball.sort_values(
    by=["bowler", "match_date", "match_id", "over_number", "ball_number"]
)

In [34]:
# BOWLER PER MATCH AGGREGATION

bowling_match = (
    ball.groupby(
        ["match_id","match_date","bowler",
         "team_bowling","team_batting","venue"]
    )
    .agg(
        balls_bowled=("ball_number","count"),
        runs_conceded=("total_runs","sum"),
        wickets=("is_wicket","sum"),
        wides=("is_wide_ball","sum"),
        noballs=("is_no_ball","sum")
    )
    .reset_index()
)

In [35]:
# CONVERT BALLS to OVERS

bowling_match["overs_bowled"] = bowling_match["balls_bowled"] / 6
# SORT TEMPORALLY

bowling_match = bowling_match.sort_values(
    by=["bowler","match_date","match_id"]
).reset_index(drop=True)

In [36]:
bowling_match["economy"] = np.where(
    bowling_match["overs_bowled"]>0,
    bowling_match["runs_conceded"]/bowling_match["overs_bowled"],
    0
)

In [37]:
# CAREER FEATURES 

bowling_match["career_matches"] = bowling_match.groupby("bowler").cumcount()
bowling_match["career_matches"] = (
    bowling_match.groupby("bowler")["career_matches"]
    .shift(1).fillna(0).astype(int)
)

bowling_match["career_wickets"] = (
    bowling_match.groupby("bowler")["wickets"]
    .cumsum().shift(1).fillna(0)
)

bowling_match["career_runs_conceded"] = (
    bowling_match.groupby("bowler")["runs_conceded"]
    .cumsum().shift(1).fillna(0)
)

bowling_match["career_overs"] = (
    bowling_match.groupby("bowler")["overs_bowled"]
    .cumsum().shift(1).fillna(0)
)

bowling_match["career_avg_wickets"] = np.where(
    bowling_match["career_matches"]>0,
    bowling_match["career_wickets"]/bowling_match["career_matches"],
    0
)

bowling_match["career_economy"] = np.where(
    bowling_match["career_overs"]>0,
    bowling_match["career_runs_conceded"]/bowling_match["career_overs"],
    0
)

In [38]:
# RECENT FORM 

bowling_match["shifted_wickets"] = bowling_match.groupby("bowler")["wickets"].shift(1)
bowling_match["shifted_runs"] = bowling_match.groupby("bowler")["runs_conceded"].shift(1)
bowling_match["shifted_overs"] = bowling_match.groupby("bowler")["overs_bowled"].shift(1)

In [39]:
# Last 5 matches wickets
bowling_match["form_wickets_last_5"] = (
    bowling_match.groupby("bowler")["shifted_wickets"]
    .rolling(5,min_periods=1)
    .mean()
    .reset_index(level=0,drop=True)
)

In [40]:
# Last 10 matches wickets
bowling_match["form_wickets_last_10"] = (
    bowling_match.groupby("bowler")["shifted_wickets"]
    .rolling(10,min_periods=1)
    .mean()
    .reset_index(level=0,drop=True)
)

In [41]:
# Recent economy last 5
def recent_economy(runs,overs,window):
    r = runs.rolling(window,min_periods=1).sum()
    o = overs.rolling(window,min_periods=1).sum()
    return np.where(o>0,r/o,0)

bowling_match["recent_economy_last_5"] = recent_economy(
    bowling_match["shifted_runs"],
    bowling_match["shifted_overs"],
    5
)

In [42]:
# OPPONENT & VENUE HISTORY

def safe_expanding(group):
    return group.shift(1).expanding().mean()

bowling_match["avg_wkts_vs_opponent"] = (
    bowling_match.groupby(["bowler","team_batting"])["wickets"]
    .transform(safe_expanding)
    .fillna(0)
)

bowling_match["avg_wkts_at_venue"] = (
    bowling_match.groupby(["bowler","venue"])["wickets"]
    .transform(safe_expanding)
    .fillna(0)
)

In [43]:
# TARGET

bowling_match["target_next_match_wickets"] = (
    bowling_match.groupby("bowler")["wickets"].shift(-1)
)

bowling_match = bowling_match[
    bowling_match["target_next_match_wickets"].notna()
].copy()

In [44]:
# FINAL FEATURE SET

drop_cols = [
    "wickets","career_wickets","career_runs_conceded",
    "career_overs","shifted_wickets","shifted_runs","shifted_overs"
]

features_df = bowling_match.drop(columns=drop_cols)
features_df = features_df.fillna(0)

In [45]:
import os
import pickle

In [46]:
# Frequency Encoding
high_card_cols = ["bowler","team_bowling","team_batting","venue"]
frequency_encoders = {}

for col in high_card_cols:
    freq = features_df[col].value_counts(normalize=True)
    features_df[col+"_freq"] = features_df[col].map(freq)
    frequency_encoders[col] = freq.to_dict()

In [47]:
# Drop original 
features_df = features_df.drop(
    columns=["bowler","team_bowling","team_batting","venue","match_id","match_date"]
)

In [48]:
os.makedirs("Datasets/Final_Features",exist_ok=True)

features_df.to_csv(
    "Datasets/Final_Features/final_bowler_dataset.csv",
    index=False
)

In [49]:
features_df.dtypes

balls_bowled                   int64
runs_conceded                  int64
wides                          int64
noballs                        int64
overs_bowled                 float64
economy                      float64
career_matches                 int64
career_avg_wickets           float64
career_economy               float64
form_wickets_last_5          float64
form_wickets_last_10         float64
recent_economy_last_5        float64
avg_wkts_vs_opponent         float64
avg_wkts_at_venue            float64
target_next_match_wickets    float64
bowler_freq                  float64
team_bowling_freq            float64
team_batting_freq            float64
venue_freq                   float64
dtype: object

In [50]:
# Save encoders
pipeline = {
    "frequency_encoders":frequency_encoders,
    "final_feature_columns":features_df.columns.tolist()
}

with open("Datasets/Final_Features/bowler_features_pipeline.pkl","wb") as f:
    pickle.dump(pipeline,f)

In [51]:
features_df.head(10)

,balls_bowled,runs_conceded,wides,noballs,overs_bowled,economy,career_matches,career_avg_wickets,career_economy,form_wickets_last_5,form_wickets_last_10,recent_economy_last_5,avg_wkts_vs_opponent,avg_wkts_at_venue,target_next_match_wickets,bowler_freq,team_bowling_freq,team_batting_freq,venue_freq
0,24,32,0,0,4.000000,8.000000,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,1.0,0.001429,0.117479,0.019329,0.013312
1,14,11,1,0,2.333333,4.714286,0,0.000000,8.000000,2.000000,2.000000,8.000000,0.0,0.000000,1.0,0.001429,0.117479,0.117329,0.062650
2,19,32,0,1,3.166667,10.105263,1,3.000000,6.789474,1.500000,1.500000,6.789474,2.0,0.000000,1.0,0.001429,0.117479,0.019329,0.006242
3,13,16,0,1,2.166667,7.384615,2,2.000000,7.894737,1.333333,1.333333,7.894737,0.0,0.000000,1.0,0.001429,0.117479,0.109131,0.042720
4,25,36,1,0,4.166667,8.640000,3,1.666667,7.800000,1.250000,1.250000,7.800000,0.0,0.000000,2.0,0.001429,0.117479,0.113568,0.053324
5,25,40,1,0,4.166667,9.600000,4,1.500000,8.021053,1.200000,1.200000,8.021053,0.0,0.000000,0.0,0.001429,0.117479,0.113718,0.041968
6,12,29,0,0,2.000000,14.500000,5,1.600000,8.350000,1.200000,1.333333,8.437500,0.0,2.000000,0.0,0.001429,0.117479,0.116576,0.041968
7,12,17,0,0,2.000000,8.500000,6,1.333333,8.909091,1.000000,1.142857,9.765957,0.0,1.000000,3.0,0.001429,0.117479,0.099880,0.041968
8,25,25,1,0,4.166667,6.000000,7,1.142857,8.875000,0.800000,1.000000,9.517241,1.0,0.666667,1.0,0.001429,0.117479,0.113568,0.041968
9,12,21,0,0,2.000000,10.500000,8,1.375000,8.449704,1.200000,1.222222,8.909091,1.5,1.250000,1.0,0.001429,0.117479,0.019329,0.041968
